In [0]:
display(dbutils.fs.ls("abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/"))

In [0]:
%sql
-- Garante que o schema existe (catalogo.schema)
CREATE SCHEMA IF NOT EXISTS bikesales.bikes;

-- Cria volumes apontando para as pastas dentro do container 'bikesales'
CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.bikes.bronze
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze';

CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.bikes.silver
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/silver';

CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.bikes.gold
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/gold';

CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.bikes.raw
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/raw';

In [0]:
# Definir pastas do projeto em variáveis para facilitar
raw_path      = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/raw/'
bronze_path   = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/silver/'
gold_path     = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/gold/'


In [0]:
# Lista os arquivos na pasta raw
display(dbutils.fs.ls(raw_path))

In [0]:
from pyspark.sql.functions import current_date

# 1. Lista com o nome exato dos arquivos
tabelas = [
    "brands", "categories", "customers", "order_items", 
    "orders", "products", "staffs", "stocks", "stores"
]

for tabela in tabelas:
    caminho_origem = f"{raw_path.strip('/')}/{tabela}.csv"
    caminho_destino = f"{bronze_path.strip('/')}/{tabela}"
    
    print(f"Processando: {tabela}.csv -> Bronze")
    
    try:
      
        df_temp = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(caminho_origem)
        
        # Adiciona a coluna de data corrente
        df_final = df_temp.withColumn("data_carga_bronze", current_date())
        
        # Gravação em formato Delta na camada Bronze
        df_final.write \
            .mode('overwrite') \
            .format('delta') \
            .option("mergeSchema", "true") \
            .save(caminho_destino)
            
        print(f" Sucesso: {tabela} carregada na Bronze.")
        
    except Exception as e:
        print(f" Erro ao processar {tabela}: {str(e)}")

print("\n Processo finalizado")

In [0]:
# Lendo um arquivo via caminho do Volume
df = spark.read.format("delta").load("/Volumes/bikesales/bikes/bronze/brands")
display(df)

In [0]:
# Listar os arquivos na pasta raw
display(dbutils.fs.ls(bronze_path))

In [0]:
# 1. Definir o caminho base da bronze
path_bronze_base = "abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze/"

# 2. Listar todas as pastas dentro de bronze
pastas = dbutils.fs.ls(path_bronze_base)

for p in pastas:
    nome_tabela = p.name.strip('/')
    caminho_completo = p.path
    
    print(f"Registrando tabela: {nome_tabela}...")
    
    try:
        # Lendo os dados Delta da pasta
        df_temp = spark.read.format("delta").load(caminho_completo)
        
        # Criando a tabela no catálogo bikesales no schema bikes

        df_temp.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"bikesales.bikes.{nome_tabela}")
            
        print(f" Tabela {nome_tabela} criada com sucesso!")
        
    except Exception as e:
        print(f" Erro ao processar {nome_tabela}: {e}")

print("\n--- Processo de registro finalizado ---")